In [ ]:
print("HIi")

HIi


In [ ]:
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input, Concatenate
from tensorflow.keras.applications import Xception, EfficientNetB0
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from google.colab import files
import os
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, f1_score, confusion_matrix


In [ ]:
REAL_PATH = "/content/Real"
FAKE_PATH = "/content/Fake"


In [ ]:
os.makedirs(REAL_PATH, exist_ok=True)
os.makedirs(FAKE_PATH, exist_ok=True)


In [ ]:
def extract_frames(video_path, num_frames=1):
    cap = cv2.VideoCapture(video_path)
    frames = []
    frame_count = 0
    while cap.isOpened() and frame_count < num_frames:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, (128, 128))  # Resize to match model input
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # Convert to RGB
        frames.append(frame)
        frame_count += 1
    cap.release()
    return np.array(frames)

In [ ]:
print("📂 Upload Real Videos")
uploaded_real = files.upload()
for filename in uploaded_real.keys():
    os.rename(filename, os.path.join(REAL_PATH, filename))

📂 Upload Real Videos


Saving id6_0001.mp4 to id6_0001.mp4
Saving id6_0002.mp4 to id6_0002.mp4
Saving id6_0003.mp4 to id6_0003.mp4
Saving id6_0004.mp4 to id6_0004.mp4
Saving id6_0005.mp4 to id6_0005.mp4
Saving id6_0006.mp4 to id6_0006.mp4
Saving id6_0007.mp4 to id6_0007.mp4
Saving id6_0008.mp4 to id6_0008.mp4
Saving id6_0009.mp4 to id6_0009.mp4
Saving id7_0000.mp4 to id7_0000.mp4
Saving id7_0001.mp4 to id7_0001.mp4
Saving id7_0002.mp4 to id7_0002.mp4
Saving id7_0003.mp4 to id7_0003.mp4
Saving id7_0004.mp4 to id7_0004.mp4
Saving id7_0005.mp4 to id7_0005.mp4
Saving id7_0006.mp4 to id7_0006.mp4
Saving id7_0007.mp4 to id7_0007.mp4
Saving id7_0008.mp4 to id7_0008.mp4
Saving id7_0009.mp4 to id7_0009.mp4
Saving id8_0000.mp4 to id8_0000.mp4
Saving id8_0001.mp4 to id8_0001.mp4
Saving id8_0002.mp4 to id8_0002.mp4
Saving id8_0003.mp4 to id8_0003.mp4
Saving id8_0004.mp4 to id8_0004.mp4


In [ ]:
print("📂 Upload Fake Videos")
uploaded_fake = files.upload()
for filename in uploaded_fake.keys():
    os.rename(filename, os.path.join(FAKE_PATH, filename))

📂 Upload Fake Videos


Saving id0_id1_0001.mp4 to id0_id1_0001.mp4
Saving id0_id1_0002.mp4 to id0_id1_0002.mp4
Saving id0_id1_0003.mp4 to id0_id1_0003.mp4
Saving id0_id1_0005.mp4 to id0_id1_0005.mp4
Saving id0_id1_0006.mp4 to id0_id1_0006.mp4
Saving id0_id1_0007.mp4 to id0_id1_0007.mp4
Saving id0_id1_0009.mp4 to id0_id1_0009.mp4
Saving id0_id2_0000.mp4 to id0_id2_0000.mp4
Saving id0_id2_0001.mp4 to id0_id2_0001.mp4
Saving id0_id2_0002.mp4 to id0_id2_0002.mp4
Saving id0_id2_0003.mp4 to id0_id2_0003.mp4
Saving id0_id2_0004.mp4 to id0_id2_0004.mp4
Saving id0_id2_0005.mp4 to id0_id2_0005.mp4
Saving id0_id2_0006.mp4 to id0_id2_0006.mp4
Saving id0_id2_0007.mp4 to id0_id2_0007.mp4
Saving id0_id2_0008.mp4 to id0_id2_0008.mp4
Saving id0_id2_0009.mp4 to id0_id2_0009.mp4
Saving id0_id3_0000.mp4 to id0_id3_0000.mp4
Saving id0_id3_0001.mp4 to id0_id3_0001.mp4
Saving id0_id3_0002.mp4 to id0_id3_0002.mp4
Saving id0_id3_0003.mp4 to id0_id3_0003.mp4
Saving id0_id3_0004.mp4 to id0_id3_0004.mp4
Saving id0_id3_0005.mp4 to id0_i

In [ ]:
# Modify data preparation to extract only the first frame
data, labels = [], []

# Process Real videos
print("Processing Real videos...")
for video_file in tqdm(os.listdir(REAL_PATH)):
    video_path = os.path.join(REAL_PATH, video_file)
    frames = extract_frames(video_path)
    if frames is not None and len(frames) > 0:
        data.append(frames[0])  # Pick only the first frame
        labels.append(0)  # Label 0 for real

# Process Fake videos
print("Processing Fake videos...")
for video_file in tqdm(os.listdir(FAKE_PATH)):
    video_path = os.path.join(FAKE_PATH, video_file)
    frames = extract_frames(video_path)
    if frames is not None and len(frames) > 0:
        data.append(frames[0])  # Pick only the first frame
        labels.append(1)  # Label 1 for fake

# Convert to numpy array
data = np.array(data)  # Shape: (num_videos, 128, 128, 3)
labels = np.array(labels)  # Shape: (num_videos,)

# Ensure the shape of data is correct
print("Data shape:", data.shape)  # Should be (num_videos, 128, 128, 3)
print("Labels shape:", labels.shape)  # Should be (num_videos,)

# Convert labels to categorical
y_train = to_categorical(labels, num_classes=2)

# Split dataset
X_train, X_temp, y_train, y_temp = train_test_split(data, labels, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Convert labels to categorical for training
y_train = to_categorical(y_train, num_classes=2)
y_val = to_categorical(y_val, num_classes=2)
y_test = to_categorical(y_test, num_classes=2)

# Check the shapes before training
print("X_train shape:", X_train.shape)  # Should be (num_train_samples, 128, 128, 3)
print("y_train shape:", y_train.shape)  # Should be (num_train_samples, 2)

Processing Real videos...


100%|██████████| 24/24 [00:00<00:00, 72.33it/s]


Processing Fake videos...


100%|██████████| 78/78 [00:00<00:00, 162.78it/s]

Data shape: (102, 128, 128, 3)
Labels shape: (102,)
X_train shape: (71, 128, 128, 3)
y_train shape: (71, 2)


In [ ]:
y_train = to_categorical(labels, num_classes=2)
X_train, X_temp, y_train, y_temp = train_test_split(data, labels, test_size=0.3, random_state=42)


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(data, labels, test_size=0.3, random_state=42)
y_train = to_categorical(y_train, num_classes=2)


In [ ]:
X_train = np.array(X_train, dtype=np.float32)
X_val = np.array(X_val, dtype=np.float32)
y_train = np.array(y_train, dtype=np.float32)
y_val = np.array(y_val, dtype=np.float32)


In [ ]:
tf.keras.backend.clear_session()


In [ ]:
# Split dataset
X_train, X_temp, y_train, y_temp = train_test_split(data, labels, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Convert labels to categorical
y_train = to_categorical(y_train, num_classes=2)
y_val = to_categorical(y_val, num_classes=2)
y_test = to_categorical(y_test, num_classes=2)

# Define the Model
input_shape = (128, 128, 3)
input_layer = Input(shape=input_shape)

# Load Pretrained Models
xception_base = Xception(weights='imagenet', include_top=False, input_tensor=input_layer)
efficientnet_base = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=input_layer)

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
input_layer = tf.keras.layers.Input(shape=(128, 128, 3))


In [ ]:
from tensorflow.keras.layers import Input, Concatenate, Dense, GlobalAveragePooling2D
from tensorflow.keras.applications import Xception, EfficientNetB7
from tensorflow.keras.models import Model

# Define input layer
input_layer = Input(shape=(128, 128, 3))

# Load pre-trained models
xception_base = Xception(weights='imagenet', include_top=False, input_tensor=input_layer)
efficientnet_base = EfficientNetB7(weights='imagenet', include_top=False, input_tensor=input_layer)

# Freeze pretrained layers
xception_base.trainable = False
efficientnet_base.trainable = False

# Feature extraction
xception_features = GlobalAveragePooling2D()(xception_base.output)
efficientnet_features = GlobalAveragePooling2D()(efficientnet_base.output)

# Concatenate features
combined_features = Concatenate()([xception_features, efficientnet_features])

# Fully Connected Layers
x = Dense(256, activation='relu')(combined_features)
x = Dense(128, activation='relu')(x)
output_layer = Dense(2, activation='softmax')(x)

# Create the ensemble model
ensemble_model = Model(inputs=input_layer, outputs=output_layer)

# Compile the model
ensemble_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Print model summary
ensemble_model.summary()


258076736/258076736 ━━━━━━━━━━━━━━━━━━━━ 9s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2             │ (None, 128, 128, 3)    │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ rescaling_2 (Rescaling)   │ (None, 128, 128, 3)    │              0 │ input_layer_2[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ normalization_1           │ (None, 128, 128, 3)    │              7 │ rescaling_2[0][0]      │
│ (Normalization)           │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ rescaling_3 (Rescaling)   │ (None, 128, 128, 3)    │              0 │ normalization_1[0][0]  │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ stem_conv_pad             │ (None, 129, 129, 3)    │              0 │ rescaling_3[0][0]      │
│ (ZeroPadding2D)           │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ stem_conv (Conv2D)        │ (None, 64, 64, 64)     │          1,728 │ stem_conv_pad[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ stem_bn                   │ (None, 64, 64, 64)     │            256 │ stem_conv[0][0]        │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ stem_activation           │ (None, 64, 64, 64)     │              0 │ stem_bn[0][0]          │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_dwconv            │ (None, 64, 64, 64)     │            576 │ stem_activation[0][0]  │
│ (DepthwiseConv2D)         │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_bn                │ (None, 64, 64, 64)     │            256 │ block1a_dwconv[0][0]   │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_activation        │ (None, 64, 64, 64)     │              0 │ block1a_bn[0][0]       │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_se_squeeze        │ (None, 64)             │              0 │ block1a_activation[0]… │
│ (GlobalAveragePooling2D)  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_se_reshape        │ (None, 1, 1, 64)       │              0 │ block1a_se_squeeze[0]… │
│ (Reshape)                 │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_se_reduce         │ (None, 1, 1, 16)       │          1,040 │ block1a_se_reshape[0]… │
│ (Conv2D)                  │                        │                │                        │
├──────────────────────

 Total params: 86,172,225 (328.72 MB)

 Trainable params: 1,213,058 (4.63 MB)

 Non-trainable params: 84,959,167 (324.09 MB)

In [ ]:
ensemble_model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10, batch_size=8)
test_loss, test_acc = ensemble_model.evaluate(X_test, y_test)


Epoch 1/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 1.0000 - loss: 5.9318e-05 - val_accuracy: 1.0000 - val_loss: 0.0052
Epoch 2/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 1.0000 - loss: 4.3267e-05 - val_accuracy: 1.0000 - val_loss: 0.0049
Epoch 3/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 19s 1s/step - accuracy: 1.0000 - loss: 3.4529e-05 - val_accuracy: 1.0000 - val_loss: 0.0047
Epoch 4/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 1.0000 - loss: 4.5740e-05 - val_accuracy: 1.0000 - val_loss: 0.0047
Epoch 5/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 1.0000 - loss: 3.7877e-05 - val_accuracy: 1.0000 - val_loss: 0.0045
Epoch 6/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step - accuracy: 1.0000 - loss: 4.7401e-05 - val_accuracy: 1.0000 - val_loss: 0.0046
Epoch 7/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 1.0000 - loss: 4.2131e-05 - val_accuracy: 1.0000 - val_loss: 0.0044
Epoch 8/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 20s 1s/step - accuracy: 1.0000 - loss: 4.7161e-05 - val_accuracy: 1.

In [ ]:
print(f"Test Accuracy: {test_acc * 100:.2f}%")

Test Accuracy: 98.67%


In [ ]:
from google.colab import files
uploaded = files.upload()
video_path = list(uploaded.keys())[0]

# Extract first frame
frames = extract_frames(video_path, num_frames=1)
if frames is None or len(frames) == 0:
    print("Error: Could not extract frames from video.")
else:
    frames = frames / 255.0  # Normalize
    frames = np.expand_dims(frames[0], axis=0)  # Add batch dimension

    # Predict using the model
    predictions = ensemble_model.predict(frames)
    class_idx = np.argmax(predictions)

    # Display result
    if class_idx == 0:
      print("✅  The video is REAL.")

    else:
      print("❌ The video is FAKE.")



Saving id0_id1_0003.mp4 to id0_id1_0003 (5).mp4
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 193ms/step
❌ The video is FAKE.


In [ ]:
# Save model as .h5
ensemble_model.save("deepfake_model.h5")

# Download the file (For Colab users)
from google.colab import files
files.download("deepfake_model.h5")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>